In [2]:
import numpy as np
from collections import Counter
from pathlib import Path
from PIL import Image

mask_dir = Path("./data/SEGTHOR_CLEAN/train/gt")

num_classes = 5

label_scale = 63

counts = Counter()
total_pixels = 0

print("Counting pixels in GT masks...")

for mask_path in mask_dir.rglob("*.png"):
    mask = np.array(Image.open(mask_path))
    # Convert pixel intensities (0, 63, 126, ...) → class indices (0, 1, 2, ...)
    mask = (mask / label_scale).astype(int)
    total_pixels += mask.size
    unique, freq = np.unique(mask, return_counts=True)
    counts.update(dict(zip(unique, freq)))

class_counts = np.array([counts.get(i, 0) for i in range(num_classes)], dtype=np.float64)
freqs = class_counts / total_pixels


inv_freqs = 1.0 / (freqs + 1e-10)
alpha = inv_freqs / inv_freqs.sum()

# Display results
print("\n Class statistics:")
for i in range(num_classes):
    print(f"Class {i}: count={class_counts[i]:.0f}, freq={freqs[i]:.6f}, alpha={alpha[i]:.6f}")

print("\n Final α vector to use in FocalLoss:")
print(alpha)


Counting pixels in GT masks...

 Class statistics:
Class 0: count=353586963, freq=0.989420, alpha=0.000182
Class 1: count=170327, freq=0.000477, alpha=0.377736
Class 2: count=2761039, freq=0.007726, alpha=0.023302
Class 3: count=126197, freq=0.000353, alpha=0.509826
Class 4: count=723282, freq=0.002024, alpha=0.088954

 Final α vector to use in FocalLoss:
[1.81959727e-04 3.77735613e-01 2.33023099e-02 5.09826454e-01
 8.89536640e-02]


In [3]:
alpha = np.array(alpha)
alpha = (alpha + 0.05) / (alpha + 0.05).sum()

In [4]:
alpha

array([0.04014557, 0.34218849, 0.05864185, 0.44786116, 0.11116293])